# 05 · Data Preprocessing

## Objective
Turn the clean dataset into model-ready inputs **without data leakage**:
1. Separate features and target
2. Create a stratified train/test split (and save it, so every later notebook uses the *same* split)
3. Define the preprocessing pipeline (scaling + encoding), fitted on training data only
4. Decide how to handle class imbalance

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can use src/churn_utils.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import inspect

import pandas as pd
from sklearn.model_selection import train_test_split

from src.churn_utils import (CATEGORICAL_FEATURES, CLEAN_DATA_PATH, NUMERIC_FEATURES,
                             RANDOM_STATE, TARGET, TEST_PATH, TRAIN_PATH, build_preprocessor)

df = pd.read_csv(CLEAN_DATA_PATH)
df.shape

(7043, 20)

## 1. Separate features and target

The target is converted to **1 = churned, 0 = retained**, the convention scikit-learn metrics expect
for the positive class.

In [2]:
X = df.drop(columns=TARGET)
y = df[TARGET].map({"No": 0, "Yes": 1})

print("X:", X.shape, "| y:", y.shape)
print("Churn rate:", round(y.mean(), 4))

X: (7043, 19) | y: (7043,)
Churn rate: 0.2654


## 2. Identify numerical and categorical features

The column lists live in `src/churn_utils.py` so all notebooks use exactly the same definitions.
Check that together they cover every feature exactly once.

In [3]:
print("Numerical  :", NUMERIC_FEATURES)
print("Categorical:", CATEGORICAL_FEATURES)
assert sorted(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == sorted(X.columns), "column lists out of sync"
print("All", X.shape[1], "features covered.")

Numerical  : ['tenure', 'MonthlyCharges', 'TotalCharges']
Categorical: ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
All 19 features covered.


## 3. Train-test split

**Decisions**
- **80 / 20 split** → ~5,600 rows to train on, ~1,400 rows for a reliable final test.
- **`stratify=y`** → both sets keep the 26.5% churn rate. Without it, random variation could give the
  test set noticeably more or fewer churners, distorting the metrics.
- **`random_state=42`** → reproducible.
- **No separate validation set:** model selection and tuning use **5-fold cross-validation on the
  training set** (notebook 07). This uses the data more efficiently than a fixed validation split, and
  the test set stays untouched until the final evaluation.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

pd.DataFrame({
    "rows": [len(X_train), len(X_test)],
    "churn_rate": [y_train.mean(), y_test.mean()],
}, index=["train", "test"]).round(4)

,rows,churn_rate
train,5634,0.2654
test,1409,0.2654


**Observation:** train (5,634 rows) and test (1,409 rows) both have a churn rate of 26.54% — stratification worked.

In [5]:
# Save the split so notebooks 06-09 load exactly the same rows
TRAIN_PATH.parent.mkdir(parents=True, exist_ok=True)
X_train.assign(**{TARGET: y_train}).to_csv(TRAIN_PATH, index=False)
X_test.assign(**{TARGET: y_test}).to_csv(TEST_PATH, index=False)
print("Saved:", TRAIN_PATH.relative_to(PROJECT_ROOT), "and", TEST_PATH.relative_to(PROJECT_ROOT))

Saved: data/processed/train.csv and data/processed/test.csv


## 4. Preprocessing pipeline

| Step | Applied to | Why |
|---|---|---|
| `StandardScaler` | `tenure`, `MonthlyCharges`, `TotalCharges` | Puts features on a comparable scale. Required for regularised logistic regression (otherwise the penalty treats features unequally); harmless for tree models. |
| `OneHotEncoder(drop="if_binary")` | 16 categorical features | Models need numbers. The categories have **no natural order** (except contract length), so one-hot is safer than label/ordinal encoding. Yes/No columns become a single 0/1 column. |
| `handle_unknown="ignore"` | categorical | A category never seen in training becomes all zeros instead of crashing at prediction time. |

No imputer is included because the cleaned data has no missing values (notebook 03).

The function lives in `src/churn_utils.py`:

In [6]:
print(inspect.getsource(build_preprocessor))

def build_preprocessor(engineered=False):
    """Scale numeric columns and one-hot encode categorical columns."""
    numeric = NUMERIC_FEATURES + (ENGINEERED_NUMERIC if engineered else [])
    categorical = CATEGORICAL_FEATURES + (ENGINEERED_CATEGORICAL if engineered else [])
    return ColumnTransformer([
        ("num", StandardScaler(), numeric),
        # drop='if_binary' turns Yes/No columns into a single 0/1 column;
        # handle_unknown='ignore' keeps the pipeline from crashing on unseen categories
        ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore",
                              sparse_output=False), categorical),
    ])



### Fit on training data only (leakage prevention)

`fit` learns statistics (means, standard deviations, category lists). If we fitted on the full dataset,
information about the test set would leak into training. So: **fit on train, only transform test.**

In notebooks 06–08 this happens automatically, because the preprocessor is a step inside a scikit-learn
`Pipeline`: during cross-validation it is re-fitted on each training fold only.

In [7]:
preprocessor = build_preprocessor()
X_train_prepared = preprocessor.fit_transform(X_train)   # learn + apply on train
X_test_prepared = preprocessor.transform(X_test)          # apply only

feature_names = preprocessor.get_feature_names_out()
print("Prepared train:", X_train_prepared.shape, "| Prepared test:", X_test_prepared.shape)
print("Number of model features:", len(feature_names))
print(list(feature_names))

Prepared train: (5634, 40) | Prepared test: (1409, 40)
Number of model features: 40
['num__tenure', 'num__MonthlyCharges', 'num__TotalCharges', 'cat__gender_Male', 'cat__SeniorCitizen_Yes', 'cat__Partner_Yes', 'cat__Dependents_Yes', 'cat__PhoneService_Yes', 'cat__MultipleLines_No', 'cat__MultipleLines_No phone service', 'cat__MultipleLines_Yes', 'cat__InternetService_DSL', 'cat__InternetService_Fiber optic', 'cat__InternetService_No', 'cat__OnlineSecurity_No', 'cat__OnlineSecurity_No internet service', 'cat__OnlineSecurity_Yes', 'cat__OnlineBackup_No', 'cat__OnlineBackup_No internet service', 'cat__OnlineBackup_Yes', 'cat__DeviceProtection_No', 'cat__DeviceProtection_No internet service', 'cat__DeviceProtection_Yes', 'cat__TechSupport_No', 'cat__TechSupport_No internet service', 'cat__TechSupport_Yes', 'cat__StreamingTV_No', 'cat__StreamingTV_No internet service', 'cat__StreamingTV_Yes', 'cat__StreamingMovies_No', 'cat__StreamingMovies_No internet service', 'cat__StreamingMovies_Yes', 

In [8]:
# Scaler statistics come from the training rows only
scaler = preprocessor.named_transformers_["num"]
pd.DataFrame({
    "scaler_mean": scaler.mean_,
    "train_mean": X_train[NUMERIC_FEATURES].mean().values,
    "full_data_mean": X[NUMERIC_FEATURES].mean().values,
}, index=NUMERIC_FEATURES).round(3)

,scaler_mean,train_mean,full_data_mean
tenure,32.485,32.485,32.371
MonthlyCharges,64.930,64.930,64.762
TotalCharges,2299.335,2299.335,2279.734


In [9]:
# After scaling: train columns have mean ~0 / std ~1; test columns are close but not exact (expected)
pd.DataFrame({
    "train_mean": X_train_prepared[:, :3].mean(axis=0),
    "train_std": X_train_prepared[:, :3].std(axis=0),
    "test_mean": X_test_prepared[:, :3].mean(axis=0),
    "test_std": X_test_prepared[:, :3].std(axis=0),
}, index=NUMERIC_FEATURES).round(3)

,train_mean,train_std,test_mean,test_std
tenure,-0.0,1.0,-0.023,0.998
MonthlyCharges,-0.0,1.0,-0.028,0.992
TotalCharges,-0.0,1.0,-0.043,0.972


**Observations**
- 19 raw features become **40 model features** (3 scaled numeric + 37 one-hot columns).
- The scaler's means equal the **training** means, not the full-data means → no test information was used.
- Scaled test columns have mean ≈ 0 and std ≈ 1 but not exactly — expected, because they were scaled with
  training statistics. That is exactly what happens with genuinely new customers in production.

## 5. Class imbalance

The positive class is 26.5% of the data — **moderately** imbalanced, not extreme.

**Decision: no resampling (no SMOTE / undersampling).** Instead:
1. Try `class_weight="balanced"` as a **hyperparameter** in notebook 07 (it makes errors on churners
   cost more during training), and keep it only if cross-validation supports it.
2. Choose the **decision threshold** on cross-validated predictions (notebook 08) to match the business
   goal of catching churners.

**Why:** with ~1,500 churners in training there is enough minority data to learn from. Class weights and
threshold tuning address the imbalance without creating synthetic customers and without an extra library.
If oversampling were used, it would have to be applied **inside** each CV training fold — never before
the split — to avoid leakage.

## Conclusion

| Decision | Choice | Reason |
|---|---|---|
| Target encoding | Yes → 1, No → 0 | Churn is the positive class |
| Split | 80/20, stratified, `random_state=42` | Keeps 26.5% churn in both sets; reproducible |
| Validation | 5-fold stratified CV on train | Efficient use of data; test kept for the final check |
| Numeric | StandardScaler | Needed for regularised linear models |
| Categorical | One-hot, binary → single column, unknown → ignore | Nominal categories; robust to unseen values |
| Imbalance | No resampling; test `class_weight` + tune threshold | Moderate imbalance; avoids synthetic data |
| Leakage | Preprocessor fitted on training data only, inside a Pipeline | Test set never influences fitted statistics |

Saved: `data/processed/train.csv` (5,634 rows) and `data/processed/test.csv` (1,409 rows).